# scDRS × scDeepID pathway attributions


## 1. Setup


In [ ]:
import os
import re
import shutil
import urllib.request

import scanpy as sc
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from scipy import stats
from statsmodels.stats.multitest import multipletests

matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
matplotlib.rcParams['font.family'] = 'DejaVu Sans'
matplotlib.rcParams['axes.grid'] = False

PROJ_DIR = '/g/data/fy54/lawhua/proj_scdeepid/exp_10k_tr_revision/bcell_human_gobp_2024_ep10_lr0.02_bs32_rentention0.0_maxgs400'
PLOT_DIR = f'{PROJ_DIR}/smr_plot_v3'
os.makedirs(PLOT_DIR, exist_ok=True)
print(f'Output dir: {PLOT_DIR}')


## 2. Load attributions


In [ ]:
attn_adata = sc.read(f'{PROJ_DIR}/attention_umap.h5ad')
pathway_attr = sc.read(f'{PROJ_DIR}/pathway_attributions.h5ad')
print(f'attn_adata: {attn_adata.shape}')
print(f'pathway_attr: {pathway_attr.shape}')

In [ ]:
pathway_attr_copy = pathway_attr.copy()
sc.pp.normalize_total(pathway_attr_copy)
sc.pp.scale(pathway_attr_copy, max_value=10)

assert (attn_adata.obs_names == pathway_attr_copy.obs_names).all(),\
    'attn_adata and pathway_attr_copy obs_names do not match — UMAP transfer would misalign rows'
pathway_attr_copy.obsm['X_umap'] = attn_adata.obsm['X_umap']

## 3. Load scDRS scores


In [ ]:
df_smr = pd.read_parquet('/g/data/fy54/analysis/tenk10k-causal/results/aggregate/tenk10k_phase1.scdrs.cell_score.tsv.parquet.gz')
df_smr_stat = pd.read_parquet('/g/data/fy54/analysis/tenk10k-causal/results/aggregate/tenk10k_phase1.scdrs.cell_mcp.tsv.parquet.gz')
print(f'cell_score shape: {df_smr.shape}')
print(f'cell_mcp shape:   {df_smr_stat.shape}')

b_cell_types = ['B_naive', 'B_intermediate', 'B_memory']
keep_cols = ['index', 'cell_type', 'umap_1', 'umap_2', 'sle']

df_smr_subset = df_smr[keep_cols].copy()
df_smr_subset = df_smr_subset[df_smr_subset['cell_type'].isin(b_cell_types)]

df_smr_stat_subst = df_smr_stat[keep_cols].copy()
df_smr_stat_subst = df_smr_stat_subst[df_smr_stat_subst['cell_type'].isin(b_cell_types)]
print(f'B-cell score rows: {len(df_smr_subset)}')
print(f'B-cell mcp   rows: {len(df_smr_stat_subst)}')


## 4. Merge cells


In [ ]:
common_cells = sorted(set(pathway_attr_copy.obs_names) & set(df_smr_subset['index']))
print(f'Number of common cells: {len(common_cells)}')

pathway_attr_subset = pathway_attr_copy[common_cells].copy()
df_smr_indexed = df_smr_subset.set_index('index')
df_smr_stat_indexed = df_smr_stat_subst.set_index('index')

pathway_attr_subset.obs['sle'] = df_smr_indexed.loc[pathway_attr_subset.obs_names, 'sle'].values
pathway_attr_subset.obs['sle_pval'] = df_smr_stat_indexed.loc[pathway_attr_subset.obs_names, 'sle'].values
pathway_attr_subset.obs['umap_1_scdrs'] = df_smr_indexed.loc[pathway_attr_subset.obs_names, 'umap_1'].values
pathway_attr_subset.obs['umap_2_scdrs'] = df_smr_indexed.loc[pathway_attr_subset.obs_names, 'umap_2'].values

print(f'SLE score range: {pathway_attr_subset.obs["sle"].min():.3f} to {pathway_attr_subset.obs["sle"].max():.3f}')


## 5. Per-pathway t-test + FDR


In [ ]:
pathway_attr_subset.obs['sle_status'] = np.where(
    pathway_attr_subset.obs['sle_pval'] < 0.05, 'SLE', 'Non-SLE'
)

p_values, pathway_names, sle_means, non_sle_means = [], [], [], []
for pathway in pathway_attr_subset.var_names:
    sle_data = pathway_attr_subset[pathway_attr_subset.obs['sle_status'] == 'SLE', pathway].X.flatten()
    non_sle_data = pathway_attr_subset[pathway_attr_subset.obs['sle_status'] == 'Non-SLE', pathway].X.flatten()
    _, p_val = stats.ttest_ind(sle_data, non_sle_data, alternative='greater')
    p_values.append(p_val)
    pathway_names.append(pathway)
    sle_means.append(sle_data.mean())
    non_sle_means.append(non_sle_data.mean())

_, fdr_values, _, _ = multipletests(p_values, method='fdr_bh')
neg_log_p = -np.log10(np.array(p_values))

results_df_raw = pd.DataFrame({
    'pathway': pathway_names,
    'neg_log_p': neg_log_p,
    'p_value': p_values,
    'fdr': fdr_values,
    'sle_mean': sle_means,
    'non_sle_mean': non_sle_means,
}).sort_values('neg_log_p', ascending=False).reset_index(drop=True)

results_df_raw.to_csv(f'{PLOT_DIR}/pathway_test_full.csv', index=False)
print(f'Total pathways tested: {len(results_df_raw)}')
print(f'FDR < 0.05:            {int((results_df_raw["fdr"] < 0.05).sum())}')
results_df_raw.head()


## 6. Build GO DAG


In [ ]:

obo_path = f'{PROJ_DIR}/go-basic.obo'
if not os.path.exists(obo_path):
    print(f'Downloading go-basic.obo to {obo_path} ...')
    req = urllib.request.Request(
        'http://purl.obolibrary.org/obo/go/go-basic.obo',
        headers={'User-Agent': 'Mozilla/5.0 (scDeepID-revision-notebook)'},
    )
    with urllib.request.urlopen(req) as resp, open(obo_path, 'wb') as out:
        shutil.copyfileobj(resp, out)
    print(f'  size: {os.path.getsize(obo_path)/1e6:.1f} MB')

def parse_obo(path):
    terms, cur = {}, None
    with open(path) as f:
        for raw in f:
            line = raw.rstrip('\n')
            if line == '[Term]':
                cur = {'parents': [], 'namespace': None, 'name': None, 'obsolete': False}
                continue
            if line.startswith('[') and line != '[Term]':
                if cur and cur.get('id'):
                    terms[cur['id']] = cur
                cur = None
                continue
            if cur is None:
                continue
            if line == '':
                if cur.get('id'):
                    terms[cur['id']] = cur
                cur = None
                continue
            if line.startswith('id: '):
                cur['id'] = line[4:]
            elif line.startswith('name: '):
                cur['name'] = line[6:]
            elif line.startswith('namespace: '):
                cur['namespace'] = line[11:]
            elif line.startswith('is_a: '):
                cur['parents'].append(line[6:].split(' ! ')[0])
            elif line.startswith('relationship: part_of '):
                cur['parents'].append(line[len('relationship: part_of '):].split(' ! ')[0])
            elif line == 'is_obsolete: true':
                cur['obsolete'] = True
    if cur and cur.get('id'):
        terms[cur['id']] = cur
    return terms

terms = parse_obo(obo_path)
G = nx.DiGraph()
for tid, t in terms.items():
    if t.get('obsolete'):
        continue
    for p in t['parents']:
        G.add_edge(p, tid)

def msigdb_name(name):
    s = re.sub(r'[^A-Za-z0-9]+', '_', name).strip('_').upper()
    return f'GOBP_{s}'

def bp_descendants(root):
    d = nx.descendants(G, root) | {root}
    return {x for x in d
            if terms.get(x, {}).get('namespace') == 'biological_process'
            and not terms.get(x, {}).get('obsolete')
            and terms.get(x, {}).get('name')}

print(f'Parsed {len(terms)} GO terms; DAG ready')


## 7. Annotate categories + rank


In [ ]:
BP_ROOT = 'GO:0008150'
MIN_PATHWAYS = 5

CMYK_PALETTE = [
    '#00AEEF',
    '#EC008C',
    '#F2C400',
    '#000000',
    '#ED1C24',
    '#00A651',
    '#0033A0',
    '#7B2CBF',
    '#F47B20',
    '#A0522D',
    '#7B7B7B',
    '#1B998B',
    '#C9184A',
    '#5A189A',
    '#FB8500',
    '#3A86FF',
]

def slugify(name):
    return re.sub(r'[^A-Za-z0-9]+', '_', name).strip('_').lower()

tested_set = set(results_df_raw['pathway'])
sig_pathway_names = set(results_df_raw.loc[results_df_raw['fdr'] < 0.05, 'pathway'])

candidate_cats = []
for cid in sorted(G.successors(BP_ROOT)):
    info = terms.get(cid, {})
    if info.get('namespace') != 'biological_process' or info.get('obsolete') or not info.get('name'):
        continue
    members = {msigdb_name(terms[d]['name']) for d in bp_descendants(cid)}
    n_tested = len(tested_set & members)
    n_sig = len(sig_pathway_names & members)
    if n_sig >= MIN_PATHWAYS:
        candidate_cats.append({'gid': cid, 'name': info['name'], 'members': members,
                               'n_tested': n_tested, 'n_sig': n_sig})

candidate_cats.sort(key=lambda c: -c['n_sig'])

CAT_LABELS = {c['gid']: slugify(c['name']) for c in candidate_cats}
CAT_NAMES  = {slugify(c['name']): c['name'] for c in candidate_cats}
CAT_COLORS = {slugify(c['name']): CMYK_PALETTE[i % len(CMYK_PALETTE)]
              for i, c in enumerate(candidate_cats)}
root_sets = {c['gid']: c['members'] for c in candidate_cats}

print(f'Top-level BP categories kept (≥{MIN_PATHWAYS} FDR-sig pathways):')
print(f'  {"GO ID":<12} {"Name":<40} {"n_sig":>6} {"n_tested":>10}  slug  (colour)')
for c in candidate_cats:
    slug = slugify(c['name'])
    print(f"  {c['gid']:<12} {c['name']:<40} {c['n_sig']:>6} {c['n_tested']:>10}  cat_{slug}  ({CAT_COLORS[slug]})")
print(f'\nKept {len(candidate_cats)} categories (palette has {len(CMYK_PALETTE)} colours)')

for c in candidate_cats:
    label = slugify(c['name'])
    results_df_raw[f'cat_{label}'] = results_df_raw['pathway'].isin(c['members'])

cat_col_names = [f'cat_{slugify(c["name"])}' for c in candidate_cats]
n_in_any  = int(results_df_raw[cat_col_names].any(axis=1).sum()) if cat_col_names else 0
n_in_none = len(tested_set) - n_in_any
print(f'\nIn ≥1 kept category: {n_in_any} / {len(tested_set)}')
print(f'In NONE of the kept categories: {n_in_none} / {len(tested_set)}')

results_df_raw.to_csv(f'{PLOT_DIR}/pathway_test_full.csv', index=False)

results_full = results_df_raw.copy()
results_full['effect'] = results_full['sle_mean'] - results_full['non_sle_mean']
results_full['neg_log_fdr'] = -np.log10(results_full['fdr'].clip(lower=1e-300))

sig_full = results_full[results_full['fdr'] < 0.05].sort_values('neg_log_fdr', ascending=False).reset_index(drop=True)
sig_full.to_csv(f'{PLOT_DIR}/pathway_sig_ranked.csv', index=False)
print(f'\nFDR<0.05 pathways: {len(sig_full)}')

for cat in CAT_LABELS.values():
    assert sig_full[f'cat_{cat}'].sum() >= MIN_PATHWAYS,\
        f'Bug: category {cat} has {sig_full[f"cat_{cat}"].sum()} sig pathways, expected ≥{MIN_PATHWAYS}'

print('\nTop 10 by -log10(fdr):')
cols_show = ['pathway', 'neg_log_fdr', 'effect', 'sle_mean', 'non_sle_mean', 'fdr'] + cat_col_names
print(sig_full[cols_show].head(10).to_string(index=False))


## 8. Per-category dumbbell


In [ ]:
cat_keys = list(CAT_COLORS.keys())
n_cats = len(cat_keys)

import math
n_per_cat = 30
ncols = 2
nrows = math.ceil(n_cats / ncols) if n_cats else 1

subsets = {
    cat: sig_full[sig_full[f'cat_{cat}']].head(n_per_cat).reset_index(drop=True)
    for cat in cat_keys
}
max_rows = max((len(s) for s in subsets.values()), default=0)

panel_w = 8.5
panel_h = max(5, max_rows * 0.25)
fig_w = panel_w * ncols
fig_h = panel_h * nrows

fig, axes = plt.subplots(nrows, ncols, figsize=(fig_w, fig_h), sharex=True, squeeze=False)
for idx, cat in enumerate(cat_keys):
    r, c = idx // ncols, idx % ncols
    ax = axes[r, c]
    sub = subsets[cat]
    n_sub = len(sub)
    for i, row in sub.iterrows():
        y_pos = n_sub - 1 - i
        ax.plot([row['non_sle_mean'], row['sle_mean']], [y_pos, y_pos], 'k-', alpha=0.5)
        ax.scatter(row['non_sle_mean'], y_pos, color='lightblue',  s=60, label='Non-SLE' if i == 0 else '')
        ax.scatter(row['sle_mean'],     y_pos, color='lightcoral', s=60, label='SLE'     if i == 0 else '')
    sub_labels = sub['pathway'].str.replace('GOBP_', '').str.replace('_', ' ').str.capitalize().tolist()[::-1]
    ax.set_yticks(range(n_sub))
    ax.set_yticklabels(sub_labels, fontsize=10)
    ax.set_ylim(-0.5, max(0.5, n_sub - 0.5))
    ax.set_title(f'{CAT_NAMES[cat]} (n={n_sub})', color=CAT_COLORS[cat], fontsize=14)
    ax.set_xlabel('Mean Attribution Score (z-scaled)', fontsize=12)
    ax.tick_params(axis='x', labelsize=11)
    ax.grid(axis='x', alpha=0.3)
    if n_sub:
        ax.legend(loc='lower right', fontsize=10, framealpha=0.9)

for idx in range(n_cats, nrows * ncols):
    r, c = idx // ncols, idx % ncols
    axes[r, c].set_visible(False)

fig.suptitle(f'Top {n_per_cat} pathways per top-level GOBP category (global FDR<0.05, ranked by -log10(FDR))',
             fontsize=15, y=1.005)
plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/top_per_category_dumbbell.pdf', dpi=300, bbox_inches='tight')
plt.show()


## 9. Top-5 biological-regulation


In [ ]:

BIO_REG_COL = 'cat_biological_regulation'
assert BIO_REG_COL in sig_full.columns,\
    f'{BIO_REG_COL} not present — was it filtered out by MIN_PATHWAYS? Inspect candidate_cats.'

pathways_to_plot = sig_full[sig_full[BIO_REG_COL]].head(5)['pathway'].tolist()
print('Top 5 biological-regulation pathways by -log10(FDR):')
for p in pathways_to_plot:
    row = sig_full[sig_full['pathway'] == p].iloc[0]
    print(f'  {p:<60}  -log10(fdr)={row["neg_log_fdr"]:>6.2f}  effect={row["effect"]:+.3f}')

color_panels = pathways_to_plot + ['celltype']
sc.settings.set_figure_params(figsize=(4.5, 4.0))
umap_titles = [p.replace('GOBP_', '').replace('_', ' ').capitalize() for p in pathways_to_plot] + ['Cell type']
fig = sc.pl.umap(
    pathway_attr_subset, color=color_panels,
    title=umap_titles,
    cmap='magma_r', size=3, ncols=3, return_fig=True, show=False,
)
for ax in fig.get_axes():
    ax.grid(False)
    for artist in ax.collections:
        artist.set_rasterized(True)
fig.savefig(f'{PLOT_DIR}/top5_biolreg_umap.pdf', dpi=300, bbox_inches='tight')
plt.show()

for pathway in pathways_to_plot:
    max_val = pathway_attr_subset[:, pathway].X.max()
    print(f'{pathway}: max attribution = {max_val:.3f}')


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(11, 8))
axes = axes.flatten()
colors = ['lightblue', 'lightcoral']

for i, pathway in enumerate(pathways_to_plot):
    axes[i].grid(False)
    sle_data = pathway_attr_subset[pathway_attr_subset.obs['sle_status'] == 'SLE', pathway].X.flatten()
    non_sle_data = pathway_attr_subset[pathway_attr_subset.obs['sle_status'] == 'Non-SLE', pathway].X.flatten()
    _, p_value_greater = stats.ttest_ind(sle_data, non_sle_data, alternative='greater')

    parts = axes[i].violinplot([non_sle_data, sle_data], positions=[1, 2],
                                showmeans=False, showmedians=False, showextrema=False)
    for j, pc in enumerate(parts['bodies']):
        pc.set_facecolor(colors[j])
        pc.set_alpha(0.6)

    axes[i].boxplot([non_sle_data, sle_data], positions=[1, 2],
                    widths=0.15, showfliers=False, patch_artist=True,
                    boxprops=dict(facecolor='white', color='black'),
                    medianprops=dict(color='black', linewidth=2))

    axes[i].set_xticks([1, 2])
    axes[i].set_xticklabels(['Non-SLE', 'SLE'], fontsize=12)
    axes[i].set_title(pathway.replace('GOBP_', '').replace('_', ' ').capitalize(), fontsize=11)
    axes[i].set_ylabel('Pathway Score', fontsize=12)

    direction = '\u2191' if sle_data.mean() > non_sle_data.mean() else '\u2193'
    axes[i].text(0.5, 0.95, f'SLE {direction}\np = {p_value_greater:.3e}',
                 transform=axes[i].transAxes, ha='center', va='top', fontsize=11,
                 bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

for k in range(len(pathways_to_plot), len(axes)):
    axes[k].set_visible(False)

plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/top5_biolreg_violin.pdf', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
line_colors = ['lightcoral', 'lightskyblue', 'lightgreen', 'peachpuff', 'plum']
decile_labels = [f'Q{i}' for i in range(1, 11)]

plt.figure(figsize=(12, 5))
sle_scores = pathway_attr_subset.obs['sle'].values
quantiles = pd.qcut(sle_scores, q=10, labels=decile_labels)

for pathway, color in zip(pathways_to_plot, line_colors):
    pathway_scores = pathway_attr_subset[:, pathway].X.flatten()
    means = [pathway_scores[quantiles == q].mean() for q in decile_labels]
    plt.plot(range(1, 11), means, marker='o', markersize=8, color=color, linewidth=2,
             label=pathway.replace('GOBP_', '').replace('_', ' ').capitalize())

plt.xlabel('scDRS Score Quantiles', fontsize=15)
plt.ylabel('Mean Pathway Score', fontsize=15)
plt.title('Top 5 biological-regulation pathways by SLE score (10 deciles)', fontsize=17)
plt.xticks(range(1, 11), decile_labels)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/top5_biolreg_decile.pdf', dpi=300, bbox_inches='tight')
plt.show()
